<a href="https://colab.research.google.com/github/Maira-26/UrduNexus-ByT5/blob/main/UrduNexus_Demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ============================================================
# UrduNexus — Self-Contained + Voice + bore.pub Tunnel
# ============================================================

import os
import subprocess
import re
import time
import socket
import threading
import nest_asyncio
import hashlib
from pathlib import Path

# --- 1. AUTO-GENERATE THE DIRECTORIES & FRONTEND ---
print("📁 Building frontend and static directories...")
os.makedirs("/content/frontend", exist_ok=True)
os.makedirs("/content/static", exist_ok=True)

html_code = """<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>UrduNexus | AI Transliteration</title>
    <link href="https://fonts.googleapis.com/css2?family=Inter:wght@400;600;800&display=swap" rel="stylesheet">
    <style>
        * { box-sizing: border-box; margin: 0; padding: 0; }
        body {
            font-family: 'Inter', -apple-system, sans-serif;
            background: linear-gradient(-45deg, #0f172a, #1e1b4b, #312e81, #0f172a);
            background-size: 400% 400%;
            animation: gradientBG 15s ease infinite;
            color: #f8fafc;
            min-height: 100vh;
            display: flex;
            flex-direction: column;
            align-items: center;
            padding: 3rem 1rem;
        }
        @keyframes gradientBG { 0% { background-position: 0% 50%; } 50% { background-position: 100% 50%; } 100% { background-position: 0% 50%; } }
        header { text-align: center; margin-bottom: 4rem; animation: fadeInDown 0.8s ease-out; }
        header h1 {
            font-size: 3.5rem; font-weight: 800;
            background: linear-gradient(to right, #38bdf8, #818cf8, #c084fc);
            -webkit-background-clip: text; -webkit-text-fill-color: transparent;
            letter-spacing: -1px; margin-bottom: 0.5rem;
        }
        header p { color: #94a3b8; font-size: 1.1rem; letter-spacing: 1px; }
        .container {
            display: flex; gap: 2rem; width: 100%; max-width: 1200px; flex-wrap: wrap;
            justify-content: center; animation: fadeInUp 1s ease-out forwards; opacity: 0; animation-delay: 0.2s;
        }
        .glass-panel {
            flex: 1; min-width: 340px; background: rgba(255, 255, 255, 0.03);
            backdrop-filter: blur(16px); -webkit-backdrop-filter: blur(16px);
            border: 1px solid rgba(255, 255, 255, 0.08); border-radius: 24px; padding: 2rem;
            box-shadow: 0 25px 50px -12px rgba(0, 0, 0, 0.5);
            transition: transform 0.4s cubic-bezier(0.175, 0.885, 0.32, 1.275), border-color 0.3s;
            display: flex; flex-direction: column;
        }
        .glass-panel:hover { transform: translateY(-8px); border-color: rgba(255, 255, 255, 0.15); }
        .panel-header {
            display: flex; justify-content: space-between; align-items: center;
            margin-bottom: 1.5rem; color: #cbd5e1; font-weight: 600; font-size: 0.85rem;
            text-transform: uppercase; letter-spacing: 2px;
        }
        textarea {
            width: 100%; height: 220px; background: transparent; border: none; color: #f1f5f9;
            font-size: 1.2rem; line-height: 1.6; resize: none; outline: none; font-family: inherit; transition: color 0.3s;
        }
        textarea::placeholder { color: rgba(148, 163, 184, 0.5); }
        #urdu-output {
            direction: rtl; font-family: 'Jameel Noori Nastaleeq', 'Nafees Web Naskh', serif;
            font-size: 2.2rem; line-height: 2; color: #fff;
        }
        .controls {
            margin-top: auto; padding-top: 1.5rem; display: flex; justify-content: flex-end;
            align-items: center; gap: 1rem; border-top: 1px solid rgba(255, 255, 255, 0.05);
        }
        button {
            background: linear-gradient(135deg, #3b82f6, #6366f1); color: white; border: none;
            padding: 0.8rem 2.2rem; border-radius: 99px; font-size: 1rem; font-weight: 600; cursor: pointer;
            transition: all 0.3s ease; box-shadow: 0 4px 15px rgba(99, 102, 241, 0.3); display: flex; align-items: center; gap: 0.5rem;
        }
        button:hover { box-shadow: 0 8px 25px rgba(99, 102, 241, 0.5); transform: translateY(-2px) scale(1.02); }
        button:active { transform: translateY(1px); }
        button:disabled { background: #334155; color: #64748b; box-shadow: none; cursor: not-allowed; transform: none; }
        .secondary-btn { background: rgba(255, 255, 255, 0.1); border: 1px solid rgba(255, 255, 255, 0.2); box-shadow: none; }
        .secondary-btn:hover { background: rgba(255, 255, 255, 0.2); box-shadow: 0 4px 15px rgba(255, 255, 255, 0.1); }
        #audio-btn { background: linear-gradient(135deg, #10b981, #059669); box-shadow: 0 4px 15px rgba(16, 185, 129, 0.3); display: none; }
        #audio-btn:hover { box-shadow: 0 8px 25px rgba(16, 185, 129, 0.5); }
        .loader {
            display: none; width: 20px; height: 20px; border: 3px solid rgba(255,255,255,0.1);
            border-top-color: #f8fafc; border-radius: 50%; animation: spin 1s linear infinite;
        }
        @keyframes spin { 100% { transform: rotate(360deg); } }
        @keyframes fadeInDown { from { opacity: 0; transform: translateY(-30px); } to { opacity: 1; transform: translateY(0); } }
        @keyframes fadeInUp { from { opacity: 0; transform: translateY(30px); } to { opacity: 1; transform: translateY(0); } }
    </style>
</head>
<body>
    <header>
        <h1>UrduNexus</h1>
        <p>AI-Powered Transliteration Engine</p>
    </header>
    <div class="container">
        <div class="glass-panel">
            <div class="panel-header">Roman Urdu Input</div>
            <textarea id="roman-input" placeholder="Type your Roman Urdu text here..."></textarea>
            <div class="controls">
                <div class="loader" id="loading-spinner"></div>
                <button id="clear-btn" class="secondary-btn" onclick="clearText()">Clear</button>
                <button id="translate-btn" onclick="translateText()">Translate</button>
            </div>
        </div>
        <div class="glass-panel">
            <div class="panel-header">Native Urdu Script</div>
            <textarea id="urdu-output" readonly placeholder="ترجمہ یہاں نظر آئے گا..."></textarea>
            <div class="controls">
                <button id="audio-btn" onclick="playAudio()">🔊 Listen</button>
            </div>
        </div>
    </div>

    <audio id="tts-player"></audio>

    <script>
        let currentAudioUrl = "";

        function clearText() {
            document.getElementById('roman-input').value = "";
            document.getElementById('urdu-output').value = "";
            document.getElementById('audio-btn').style.display = 'none';
            document.getElementById('tts-player').src = "";
        }

        async function translateText() {
            const inputText = document.getElementById('roman-input').value.trim();
            if (!inputText) return;
            const translateBtn = document.getElementById('translate-btn');
            const clearBtn = document.getElementById('clear-btn');
            const audioBtn = document.getElementById('audio-btn');
            const loadingSpinner = document.getElementById('loading-spinner');
            const urduOutput = document.getElementById('urdu-output');

            translateBtn.disabled = true; clearBtn.disabled = true;
            translateBtn.innerText = "Processing..."; loadingSpinner.style.display = 'block';
            urduOutput.value = ""; audioBtn.style.display = 'none';

            try {
                const response = await fetch('/translate-and-speak', {
                    method: 'POST', headers: { 'Content-Type': 'application/json' },
                    body: JSON.stringify({ roman_urdu: inputText })
                });
                if (!response.ok) throw new Error(`Server Error: ${response.status}`);
                const data = await response.json();

                urduOutput.value = data.urdu_text || "Translation empty.";

                if (data.audio_file) {
                    currentAudioUrl = data.audio_file + "?t=" + Date.now();
                    document.getElementById("tts-player").src = currentAudioUrl;
                    audioBtn.style.display = "flex";
                }
            } catch (error) {
                console.error("Translation Error:", error); urduOutput.value = "Error: Connection interrupted.";
            } finally {
                setTimeout(() => {
                    translateBtn.disabled = false; clearBtn.disabled = false;
                    translateBtn.innerText = "Translate"; loadingSpinner.style.display = 'none';
                }, 100);
            }
        }

        function playAudio() {
            document.getElementById('tts-player').play();
        }
    </script>
</body>
</html>"""

with open("/content/frontend/index.html", "w", encoding="utf-8") as f:
    f.write(html_code)
print("✅ index.html automatically created!")


# --- 2. INSTALL DEPENDENCIES & TUNNEL (Added edge-tts) ---
print("📦 Installing backend dependencies...")
subprocess.run(["pip", "install", "-q", "fastapi", "uvicorn[standard]", "transformers", "nest-asyncio", "sentencepiece", "torch", "edge-tts"], check=True)

subprocess.run(["wget", "-q", "https://github.com/ekzhang/bore/releases/download/v0.5.0/bore-v0.5.0-x86_64-unknown-linux-musl.tar.gz", "-O", "/tmp/bore.tar.gz"], check=True)
subprocess.run(["tar", "-xzf", "/tmp/bore.tar.gz", "-C", "/usr/local/bin/"], check=True)
subprocess.run(["chmod", "+x", "/usr/local/bin/bore"], check=True)

# --- 3. LOAD MODEL & LOGIC ---
import torch
import edge_tts
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import uvicorn
from fastapi import FastAPI, HTTPException
from fastapi.responses import FileResponse
from fastapi.staticfiles import StaticFiles
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel

MODEL_NAME = "Maira26/UrduNexus-ByT5"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🔧 Loading model on: {DEVICE.upper()}...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME, dtype=torch.float16 if DEVICE == "cuda" else torch.float32
).to(DEVICE)
model.eval()
print("✅ Model loaded.")

MAX_CHUNK_CHARS = 180
def split_into_chunks(text):
    text = text.strip()
    if not text: return []
    sentences = re.split(r'(?<=[.!?])\s+', text)
    chunks, current = [], ""
    for sentence in sentences:
        if len(sentence) > MAX_CHUNK_CHARS:
            for part in re.split(r'(?<=,)\s+', sentence):
                while len(part) > MAX_CHUNK_CHARS:
                    chunks.append(part[:MAX_CHUNK_CHARS].strip())
                    part = part[MAX_CHUNK_CHARS:]
                if current and len(current) + 1 + len(part) > MAX_CHUNK_CHARS:
                    chunks.append(current.strip()); current = part
                else:
                    current = (current + " " + part).strip() if current else part
        else:
            if current and len(current) + 1 + len(sentence) > MAX_CHUNK_CHARS:
                chunks.append(current.strip()); current = sentence
            else:
                current = (current + " " + sentence).strip() if current else sentence
    if current: chunks.append(current.strip())
    return chunks

def transliterate(roman_text):
    chunks = split_into_chunks(roman_text)
    if not chunks: return ""
    results = []
    with torch.no_grad():
        for chunk in chunks:
            inputs = tokenizer(chunk, return_tensors="pt", padding=True, truncation=True, max_length=512).to(DEVICE)
            outputs = model.generate(
                **inputs,
                max_new_tokens=512,
                num_beams=1,
                do_sample=False,
                use_cache=True
            )
            results.append(tokenizer.decode(outputs[0], skip_special_tokens=True))
    return " ".join(results)

async def generate_audio(urdu_text: str) -> str:
    text_hash = hashlib.md5(urdu_text.encode("utf-8")).hexdigest()[:12]
    filename = f"speech_{text_hash}.mp3"
    filepath = os.path.join("/content/static", filename)

    if not os.path.exists(filepath):
        communicate = edge_tts.Communicate(urdu_text, voice="ur-PK-UzmaNeural")
        await communicate.save(filepath)

    return f"/static/{filename}"

# --- 4. FASTAPI SERVER ---
app = FastAPI()
FRONTEND_DIR = Path("/content/frontend")

app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_credentials=True, allow_methods=["*"], allow_headers=["*"])

# Mount the static folder so the frontend can access the audio files
app.mount("/static", StaticFiles(directory="/content/static"), name="static")

class TranslateRequest(BaseModel):
    roman_urdu: str

@app.api_route("/", methods=["GET", "HEAD", "OPTIONS"])
async def serve_index():
    index_path = FRONTEND_DIR / "index.html"
    return FileResponse(str(index_path), media_type="text/html")

@app.post("/translate-and-speak")
async def translate_and_speak(req: TranslateRequest):
    if not req.roman_urdu.strip(): raise HTTPException(status_code=400, detail="Empty input.")
    try:
        urdu_script = transliterate(req.roman_urdu)
        audio_url = ""
        if urdu_script:
            audio_url = await generate_audio(urdu_script)

        return {"urdu_text": urdu_script, "audio_file": audio_url}
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

# --- 5. START SERVER & TUNNEL ---
PORT = 8000
subprocess.run(["fuser", "-k", f"{PORT}/tcp"], capture_output=True)
nest_asyncio.apply()

threading.Thread(
    target=lambda: uvicorn.run(app, host="0.0.0.0", port=PORT, log_level="warning"),
    daemon=True
).start()

time.sleep(3) # Wait for Uvicorn
bore_proc = subprocess.Popen(["bore", "local", str(PORT), "--to", "bore.pub"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)

print("⏳ Starting bore tunnel...")
public_url = None
for line in bore_proc.stdout:
    match = re.search(r'bore\.pub:(\d+)', line)
    if match:
        public_url = f"http://bore.pub:{match.group(1)}"
        break

if public_url:
    print("\n" + "="*60)
    print(f"  🌐 YOUR APP IS LIVE WITH VOICE:\n")
    print(f"      {public_url}\n")
    print("="*60)
else:
    print("❌ Could not get bore URL.")

📁 Building frontend and static directories...
✅ index.html automatically created!
📦 Installing backend dependencies...
🔧 Loading model on: CUDA...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/170 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


✅ Model loaded.
⏳ Starting bore tunnel...

  🌐 YOUR APP IS LIVE WITH VOICE:

      http://bore.pub:41028

